# Viz — Plotly 3D (optional)

`pip install plotly`. Subsample + optional tiny edge list.


### Interactive 3D view and a tiny edge subset

**Method:** Plotly draws all stereographic points (can be heavy) plus line segments for up to 120 edges on a small random induced subgraph of `G`.

**How to read the output:** Rotate/zoom to inspect local geometry. Edges are only a **sparse sample**—do not infer global connectivity from this view. Requires `plotly`; if the notebook errors, `pip install plotly`.


In [ ]:
# pip install plotly
from pathlib import Path
import numpy as np
import pandas as pd

import dmercator3d_io as dm
from ball_projection import stereographic_s3_to_r3

merged = dm.load_merged_parquet(Path("cache/merged.parquet"))
paths = dm.paths_for_run("d3")
G = dm.load_edges_graph(paths["edge"])
U = dm.normalize_direction_nd(merged)
name_to_i = {str(v): i for i, v in enumerate(merged["Vertex"])}
rng = np.random.default_rng(1)
verts = list(G.nodes())
sub = set(rng.choice(verts, size=min(80, len(verts)), replace=False))
subG = G.subgraph(sub).copy()
x1, x2, x3, x4 = U[:, 0], U[:, 1], U[:, 2], U[:, 3]
X, Y, Z = stereographic_s3_to_r3(x1, x2, x3, x4)

import plotly.graph_objects as go
trace = go.Scatter3d(x=X, y=Y, z=Z, mode="markers", marker=dict(size=2, opacity=0.25), name="vertices")
fig = go.Figure(data=[trace])
for u, v in list(subG.edges())[:120]:
    su, sv = str(u), str(v)
    if su not in name_to_i or sv not in name_to_i:
        continue
    i, j = name_to_i[su], name_to_i[sv]
    fig.add_trace(
        go.Scatter3d(
            x=[X[i], X[j]],
            y=[Y[i], Y[j]],
            z=[Z[i], Z[j]],
            mode="lines",
            line=dict(width=2, color="orange"),
            showlegend=False,
            opacity=0.5,
        )
    )
fig.update_layout(title="Subsample + chord edges (stereographic R³)")
fig.show()
